In [1]:
# 04_simulation_verification.ipynb
# Discrete-event verification of the Stage-6 review queue using the fixed scenario mixture in
# data/queue_params.json (s_fast=1.33 h, s_slow=8.0 h, prime share 0.60; E[S]=4.0 h, CV=0.816).
# Protocol for every check: 40 replications (seeds 0-39) x 20,000 jobs after a 2,000-job warm-up.

## Simulator and analytical references

In [2]:
import numpy as np, math, json
qp = json.load(open("../data/queue_params.json"))
p = qp["client_mix"]["prime_share"]
sf = qp["service_time_type_means"]["s_fast_prime_hours"]
ss = qp["service_time_type_means"]["s_slow_scarce_hours"]
ES = p*sf + (1-p)*ss; ES2 = p*sf**2 + (1-p)*ss**2; cs2 = (ES2 - ES**2) / ES**2
R, N, W = 40, 20000, 2000

def simulate_mgc(lam, s_fast, s_slow, p, c, n_jobs=N, warmup=W, seed=0):
    """Mean waiting time in an M/G/c queue with a two-point service-time mixture (FCFS, earliest-free server)."""
    rng = np.random.default_rng(seed)
    server_free = [0.0] * c; t = 0.0; waits = []
    for i in range(n_jobs + warmup):
        t += rng.exponential(1.0 / lam)
        svc = s_fast if rng.random() < p else s_slow
        j = int(np.argmin(server_free))
        start = max(t, server_free[j]); server_free[j] = start + svc
        if i >= warmup: waits.append(start - t)
    return float(np.mean(waits))

def pk_wq(lam): rho = lam*ES; return lam*ES2 / (2*(1 - rho))
def erlang_c_wq(lam, mu, c):
    a = lam/mu; rho = a/c
    s = sum(a**n/math.factorial(n) for n in range(c)); last = a**c/(math.factorial(c)*(1-rho))
    return last/(s+last)/(c*mu - lam)
def klb_wq(lam, c): return erlang_c_wq(lam, 1/ES, c)*(1 + cs2)/2
def ci(reps): m = float(np.mean(reps)); h = 1.96*float(np.std(reps, ddof=1))/math.sqrt(len(reps)); return m, h
print(f"E[S]={ES:.3f} h  E[S^2]={ES2:.3f}  C_S^2={cs2:.3f}")

E[S]=3.998 h  E[S^2]=26.661  C_S^2=0.668


## M/G/1 (exact Pollaczek-Khinchine) and M/G/c (c = 12)

In [3]:
rows1, rows2 = [], []
for rho in (0.4, 0.6, 0.8):
    lam = rho/ES
    m, h = ci([simulate_mgc(lam, sf, ss, p, 1, seed=s) for s in range(R)])
    rows1.append(dict(rho=rho, lam=lam, wq_pk=pk_wq(lam), wq_sim=m, ci_half=h, err_pct=100*(m - pk_wq(lam))/pk_wq(lam),
                      pk_in_ci=bool(m - h <= pk_wq(lam) <= m + h)))
    print(rows1[-1])
c = 12
for rho in (0.4, 0.6, 0.8):
    lam = rho*c/ES
    m, h = ci([simulate_mgc(lam, sf, ss, p, c, seed=s) for s in range(R)])
    rows2.append(dict(rho=rho, lam=lam, wq_sim=m, ci_half=h, wq_klb=klb_wq(lam, c)))
    print(rows2[-1])
json.dump(dict(ES=ES, ES2=ES2, cs2=cs2, s_fast=sf, s_slow=ss, p=p, c=c, reps=R, n_jobs=N, warmup=W,
               mg1_exact=rows1, mgc=rows2), open("../results/tables/sim_results.json", "w"), indent=2)

{'rho': 0.4, 'lam': 0.10005002501250625, 'wq_pk': 2.222889778222445, 'wq_sim': 2.223010862307015, 'ci_half': 0.015430491818193835, 'err_pct': 0.0054471474814570976, 'pk_in_ci': True}


{'rho': 0.6, 'lam': 0.15007503751875936, 'wq_pk': 5.0015020010005005, 'wq_sim': 5.005652982893727, 'ci_half': 0.04798323125328968, 'err_pct': 0.0829947062381664, 'pk_in_ci': True}


{'rho': 0.8, 'lam': 0.2001000500250125, 'wq_pk': 13.337338669334672, 'wq_sim': 13.251424361149654, 'ci_half': 0.24720162146925373, 'err_pct': -0.6441638044518785, 'pk_in_ci': True}


{'rho': 0.4, 'lam': 1.2006003001500751, 'wq_sim': 0.0022130829521366654, 'ci_half': 0.00024775700555407714, 'wq_klb': 0.001983093139595015}


{'rho': 0.6, 'lam': 1.8009004502251122, 'wq_sim': 0.05720928577160702, 'ci_half': 0.0022759237678512956, 'wq_klb': 0.051874163296525874}


{'rho': 0.8, 'lam': 2.4012006003001503, 'wq_sim': 0.5268926384989144, 'ci_half': 0.018482237213275424, 'wq_klb': 0.5124337652246548}


## Verification figure

In [4]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi":600,"font.size":11,"axes.edgecolor":"0.2","axes.linewidth":0.8,"grid.color":"0.85"})
def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"../results/figures/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
r1 = rows1; r2 = rows2
ax[0].plot([r["rho"] for r in r1], [r["wq_pk"] for r in r1], "o-", color="0.15", lw=1.4, label="P-K exact", markersize=7)
ax[0].errorbar([r["rho"] for r in r1], [r["wq_sim"] for r in r1], yerr=[r["ci_half"] for r in r1], fmt="s--",
               color="0.55", lw=1.2, capsize=3, markerfacecolor="white", label="Simulation (95% CI)")
ax[0].set_xlabel(r"Traffic intensity $\rho$"); ax[0].set_ylabel(r"Mean waiting time $W_q$ (h)")
ax[0].set_title("M/G/1 (exact, $c=1$)", fontsize=11); ax[0].legend(frameon=True, edgecolor="0.5")
ax[1].errorbar([r["rho"] for r in r2], [r["wq_sim"] for r in r2], yerr=[r["ci_half"] for r in r2], fmt="s-",
               color="0.15", lw=1.4, capsize=3, markerfacecolor="white", label="Simulation (95% CI)")
ax[1].plot([r["rho"] for r in r2], [r["wq_klb"] for r in r2], "o:", color="0.55", lw=1.2, label="KLB (reference)")
ax[1].set_xlabel(r"Traffic intensity $\rho$"); ax[1].set_ylabel(r"Mean waiting time $W_q$ (h)")
ax[1].set_title("M/G/c ($c=12$)", fontsize=11); ax[1].set_yscale("log"); ax[1].legend(frameon=True, edgecolor="0.5")
plt.tight_layout(); save(fig, "fig6_sim_verification"); print("fig6 saved")

fig6 saved
